In [1]:
import duckdb
from pathlib import Path
conn = duckdb.connect(Path(r"C:\Users\eddiec11us\dev_apps\customer-matching-app\src\data\db.duckdb"))

In [ ]:
query = """
WITH parentless AS (
SELECT
  v.vendor_customer_id,
  v.vendor_name,
  v.raw_vendor_customer_name,
  v.normalized_vendor_customer_name,
  v.raw_billing_zip,
  v.normalized_billing_zip,
  v.billing_state,
  v.period_date,
  v.first3_token
FROM vendor_customers v
WHERE NOT EXISTS (
  SELECT 1 
  FROM vendor_customer_to_parent_account_map accepted
  WHERE v.vendor_customer_id = accepted.vendor_customer_id
  )
), potential_siblings_token_and_zip AS (
SELECT
  ps.vendor_customer_id AS left_vendor_customer_id,
  ps.vendor_name AS left_vendor_name,
  ps.raw_vendor_customer_name AS left_raw_vendor_customer_name,
  ps.normalized_vendor_customer_name AS left_normalized_vendor_customer_name,
  ps.raw_billing_zip AS left_raw_billing_zip,
  ps.normalized_billing_zip AS left_normalized_billing_zip,
  ps.billing_state AS left_billing_state,
  ps.first3_token AS left_first3_token,
  siblings.vendor_customer_id AS right_vendor_customer_id,
  siblings.vendor_customer_id AS right_vendor_customer_id,
  siblings.vendor_name AS right_vendor_name,
  siblings.raw_vendor_customer_name AS right_raw_vendor_customer_name,
  siblings.normalized_vendor_customer_name AS right_normalized_vendor_customer_name,
  siblings.raw_billing_zip AS right_raw_billing_zip,
  siblings.normalized_billing_zip AS right_normalized_billing_zip,
  siblings.billing_state AS right_billing_state,
  siblings.first3_token AS right_first3_token,
  'token_zip' AS "match_type"
FROM parentless ps
JOIN vendor_customers siblings ON
  ps.first3_token = siblings.first3_token
  AND ps.normalized_billing_zip = siblings.normalized_billing_zip
WHERE 
  ps.vendor_customer_id < siblings.vendor_customer_id
), potential_siblings_token_only AS (
SELECT
  ps.vendor_customer_id AS left_vendor_customer_id,
  ps.vendor_name AS left_vendor_name,
  ps.raw_vendor_customer_name AS left_raw_vendor_customer_name,
  ps.normalized_vendor_customer_name AS left_normalized_vendor_customer_name,
  ps.raw_billing_zip AS left_raw_billing_zip,
  ps.normalized_billing_zip AS left_normalized_billing_zip,
  ps.billing_state AS left_billing_state,
  ps.first3_token AS left_first3_token,
  siblings.vendor_customer_id AS right_vendor_customer_id,
  siblings.vendor_customer_id AS right_vendor_customer_id,
  siblings.vendor_name AS right_vendor_name,
  siblings.raw_vendor_customer_name AS right_raw_vendor_customer_name,
  siblings.normalized_vendor_customer_name AS right_normalized_vendor_customer_name,
  siblings.raw_billing_zip AS right_raw_billing_zip,
  siblings.normalized_billing_zip AS right_normalized_billing_zip,
  siblings.billing_state AS right_billing_state,
  siblings.first3_token AS right_first3_token,
  'token_only' AS "match_type"
FROM parentless ps
JOIN vendor_customers siblings ON
  ps.first3_token = siblings.first3_token
WHERE 
  ps.vendor_customer_id < siblings.vendor_customer_id
), unioned AS (
SELECT * FROM potential_siblings_token_and_zip

UNION

SELECT * FROM potential_siblings_token_only

)
SELECT * FROM unioned
ORDER BY left_vendor_customer_id ASC, 
"""

In [21]:
df = conn.sql(query=query).df()
print(len(df))
df

1254


,left_vendor_customer_id,left_vendor_name,left_raw_vendor_customer_name,left_normalized_vendor_customer_name,left_raw_billing_zip,left_normalized_billing_zip,left_billing_state,left_first3_token,right_vendor_customer_id,right_vendor_customer_id_1,right_vendor_name,right_raw_vendor_customer_name,right_normalized_vendor_customer_name,right_raw_billing_zip,right_normalized_billing_zip,right_billing_state,right_first3_token,match_type
0,4,ACCUTECH,ACCU-TECH CORPORATION,accutech corporation,30009,30009,GA,acc,1112,1112,EXERTIS ALMO,Accu-tech Corp,accutech,30009,30009,GA,acc,token_zip
1,9,ACCUTECH,BLUESTONE,bluestone,19803,19803,DE,blu,11,11,ACCUTECH,BLUESTONE COMMUNICATIONS INC.,bluestone communications,19803,19803,DE,blu,token_zip
2,9,ACCUTECH,BLUESTONE,bluestone,19803,19803,DE,blu,10,10,ACCUTECH,BLUESTONE COMMUNICATIONS,bluestone communications,19803,19803,DE,blu,token_zip
3,10,ACCUTECH,BLUESTONE COMMUNICATIONS,bluestone communications,19803,19803,DE,blu,11,11,ACCUTECH,BLUESTONE COMMUNICATIONS INC.,bluestone communications,19803,19803,DE,blu,token_zip
4,18,ACCUTECH,CLARITY ITS,clarity its,30114,30114,GA,cla,19,19,ACCUTECH,CLARITY ITS SOLUTIONS,clarity its solutions,30114,30114,GA,cla,token_zip
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1249,3038,SYNNEX,TELE-COMMUNICATION INC DBA UNIFIEDCOMMUNICATIONS,telecommunication dba unifiedcommunications,77092,77092,TX,tel,3039,3039,SYNNEX,TELE-COMMUNICATION INC DBA UNIFIEDCOMMUNICATIO...,telecommunication dba unifiedcommunicationscom,77092,77092,TX,tel,token_zip
1250,3088,SYNNEX,VIDEOLINK INC,videolink,80207,80207,CO,vid,3089,3089,SYNNEX,"VIDEOLINK, INC.",videolink,80207,80207,CO,vid,token_zip
1251,3101,SYNNEX,WESTEND ENTERPRISES,westend enterprises,94566,94566,CA,wes,3102,3102,SYNNEX,"WESTEND ENTERPRISES UNLIMITED, INC.",westend enterprises unlimited,94566,94566,CA,wes,token_zip
1252,3246,SYNNEX CN,MICROSERVE BUSINESS COMPUTER SERVICES,microserve business computer services,V5G 4G3,V5G4G3,BC,mic,3247,3247,SYNNEX CN,MICROSERVE BUSINESS COMPUTER SERVICES / 341234...,microserve business computer services 341234 bc,V5G 4G3,V5G4G3,BC,mic,token_zip


In [ ]:
import pandas as pd
df = pd.DataFrame({"id_a": [1,2], "id_b": [2,1], "parent_id": [3, 4], "match_type": ['token_zip', 'zip_only']})



def calculate_sort_value(row: pd.Series):
    match_type = row["match_type"]
    has_parent = pd.notna(row["parent_id"])
    
    match (match_type, has_parent):
        case ("token_zip", True):
            return 1
    
        case ("token_zip", False):
            return 2
        
        case ("zip_only", True):
            return 3
    
        case ("zip_only", False):
            return 4
    
        case ("token_only", True):
            return 5
    
        case ("token_only", False):
            return 6
    



df["sort_index"] = df.apply(calculate_sort_value, axis=1)

df


,id_a,id_b,parent_id,match_type,sort_index
0,1,2,3,token_zip,1
1,2,1,4,zip_only,2
